# CLAMP BP - Timing (3 runs)

**Environment:** `clamp-analyses`  

Measures SVD, CLAMPbase and CLAMPfull times separately, 3 runs each.

In [ ]:
library(bigstatsr)
library(here)
library(CLAMP)
library(PCAtools)

source(here("config.R"))

In [2]:
input_dir <- config$GTEx$OUTPUT_DIR
output_dir <- here("output/model_performance/gtex")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

N_RUNS <- 3
N_CORES <- config$GTEx$N_CORES
base_seed <- config$GTEx$RANDOM_SVD_SEED
seeds <- base_seed + 0:(N_RUNS - 1)

In [3]:
gtex_fbm_filt <- readRDS(file.path(input_dir, "gtex_fbm_filt.rds"))
gtex_genes <- readRDS(file.path(input_dir, "gtex_genes.rds"))

n_genes <- nrow(gtex_fbm_filt)
n_samples <- ncol(gtex_fbm_filt)
SVD_K <- (min(n_genes, n_samples) - 1) %/% 4

message("Data: ", n_genes, " genes x ", n_samples, " samples")
message("SVD K = ", SVD_K)

Data: 21613 genes x 17382 samples

SVD K = 4345



In [4]:
gtex_gmtList <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)

for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12116 genes in the intersection between data and prior

Removing 2020 pathways



## SVD (3 runs)

In [ ]:
SVD_times <- numeric(N_RUNS)
svd_results <- list()
CLAMP_K_values <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("SVD run", i, "of", N_RUNS, "- seed:", seeds[i], "\n")
  
  set.seed(seeds[i])
  
  start_time <- Sys.time()
  
  svd_result <- rsvd::rsvd(gtex_fbm_filt[,], k = SVD_K)
  
  end_time <- Sys.time()
  
  # Remove NaN values
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  svd_results[[i]] <- svd_result
  
  # Estimate CLAMP K via Gavish-Donoho
  eigenvalues <- sort(svd_result$d^2 / (n_samples - 1), decreasing = TRUE)
  noise_gd    <- median(eigenvalues)
  CLAMP_K_values[i] <- PCAtools::chooseGavishDonoho(
    .dim          = c(n_genes, n_samples),
    var.explained = eigenvalues,
    noise         = noise_gd
  ) * 2
  
  SVD_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", SVD_times[i], "minutes, CLAMP_K =", CLAMP_K_values[i], "\n\n")
}

In [7]:
SVD_time_minutes <- SVD_times
names(SVD_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(SVD_time_minutes, file.path(output_dir, "SVD_time_minutes.rds"))
cat("SVD times:", SVD_time_minutes, "minutes\n")

SVD times: 23.05944 22.96758 23.64484 minutes


## CLAMPbase (3 runs)

In [8]:
CLAMPbase_times <- numeric(N_RUNS)
base_results <- list()

for (i in 1:N_RUNS) {
  cat("CLAMPbase run", i, "of", N_RUNS, "\n")
  
  set.seed(seeds[i])
  
  start_time <- Sys.time()
  
  gtex_baseRes <- CLAMPbase(
    Y      = gtex_fbm_filt,
    svdres = svd_results[[i]],
    trace  = TRUE,
    clamp_k = CLAMP_K_values[i]
  )
  
  end_time <- Sys.time()
  
  base_results[[i]] <- gtex_baseRes
  CLAMPbase_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", CLAMPbase_times[i], "minutes\n\n")
}

CLAMPbase run 1 of 3 


****

CLAMP k is set to 412

L1 is set to 45.277637028454

L2 is set to 135.832911085362

Progress 1 / 200 | Bdiff=0.222226, minCor=0.616273

Progress 2 / 200 | Bdiff=0.030933, minCor=0.915387

Progress 3 / 200 | Bdiff=0.014314, minCor=0.958798

Progress 4 / 200 | Bdiff=0.009708, minCor=0.976382

Progress 5 / 200 | Bdiff=0.007364, minCor=0.983585

Progress 6 / 200 | Bdiff=0.005958, minCor=0.987159

Progress 7 / 200 | Bdiff=0.005016, minCor=0.988719

Progress 8 / 200 | Bdiff=0.004328, minCor=0.989041

Progress 9 / 200 | Bdiff=0.003798, minCor=0.989754



Progress 10 / 200 | Bdiff=0.003379, minCor=0.990696

Progress 11 / 200 | Bdiff=0.003043, minCor=0.992362

Progress 12 / 200 | Bdiff=0.002777, minCor=0.994038

Progress 13 / 200 | Bdiff=0.002561, minCor=0.994885

Progress 14 / 200 | Bdiff=0.002384, minCor=0.995539

Progress 15 / 200 | Bdiff=0.002234, minCor=0.995312

Progress 16 / 200 | Bdiff=0.002102, minCor=0.995278

Progress 17 / 200 | Bdiff=0.001984, minCor=0.995571

Progress 18 / 200 | Bdiff=0.001874, minCor=0.996147

Progress 19 / 200 | Bdiff=0.001771, minCor=0.996422

Progress 20 / 200 | Bdiff=0.050835, minCor=0.893361

Progress 21 / 200 | Bdiff=0.016156, minCor=0.968843

Progress 22 / 200 | Bdiff=0.011613, minCor=0.978856

Progress 23 / 200 | Bdiff=0.009693, minCor=0.965359

Progress 24 / 200 | Bdiff=0.008391, minCor=0.968245

Progress 25 / 200 | Bdiff=0.007335, minCor=0.972657

Progress 26 / 200 | Bdiff=0.006493, minCor=0.977813

Progress 27 / 200 | Bdiff=0.005784, minCor=0.969459

Progress 28 / 200 | Bdiff=0.005098, minCor=0.9

Run 1 time: 2.988649 minutes

CLAMPbase run 2 of 3 


****

CLAMP k is set to 412

L1 is set to 45.2776576560399

L2 is set to 135.83297296812

Progress 1 / 200 | Bdiff=0.222226, minCor=0.616273

Progress 2 / 200 | Bdiff=0.030933, minCor=0.915385

Progress 3 / 200 | Bdiff=0.014314, minCor=0.958803

Progress 4 / 200 | Bdiff=0.009708, minCor=0.976383

Progress 5 / 200 | Bdiff=0.007364, minCor=0.983586

Progress 6 / 200 | Bdiff=0.005958, minCor=0.987160

Progress 7 / 200 | Bdiff=0.005016, minCor=0.988718

Progress 8 / 200 | Bdiff=0.004328, minCor=0.989040

Progress 9 / 200 | Bdiff=0.003798, minCor=0.989754

Progress 10 / 200 | Bdiff=0.003379, minCor=0.990697

Progress 11 / 200 | Bdiff=0.003044, minCor=0.992364

Progress 12 / 200 | Bdiff=0.002777, minCor=0.994038

Progress 13 / 200 | Bdiff=0.002561, minCor=0.994884

Progress 14 / 200 | Bdiff=0.002384, minCor=0.995539

Progress 15 / 200 | Bdiff=0.002234, minCor=0.995313

Progress 16 / 200 | Bdiff=0.002102, minCor=0.995278

Progress 17 / 200 | Bdiff=0.001984, minCor=0.995572

Progress 18 / 200 

Run 2 time: 2.829937 minutes

CLAMPbase run 3 of 3 


****

CLAMP k is set to 412

L1 is set to 45.2776247586925

L2 is set to 135.832874276078

Progress 1 / 200 | Bdiff=0.222226, minCor=0.616278

Progress 2 / 200 | Bdiff=0.030933, minCor=0.915388

Progress 3 / 200 | Bdiff=0.014314, minCor=0.958799

Progress 4 / 200 | Bdiff=0.009708, minCor=0.976382

Progress 5 / 200 | Bdiff=0.007364, minCor=0.983585

Progress 6 / 200 | Bdiff=0.005958, minCor=0.987159

Progress 7 / 200 | Bdiff=0.005016, minCor=0.988720

Progress 8 / 200 | Bdiff=0.004328, minCor=0.989043

Progress 9 / 200 | Bdiff=0.003798, minCor=0.989757

Progress 10 / 200 | Bdiff=0.003379, minCor=0.990698

Progress 11 / 200 | Bdiff=0.003043, minCor=0.992364

Progress 12 / 200 | Bdiff=0.002777, minCor=0.994039

Progress 13 / 200 | Bdiff=0.002561, minCor=0.994885

Progress 14 / 200 | Bdiff=0.002384, minCor=0.995538

Progress 15 / 200 | Bdiff=0.002234, minCor=0.995312

Progress 16 / 200 | Bdiff=0.002102, minCor=0.995278

Progress 17 / 200 | Bdiff=0.001984, minCor=0.995572

Progress 18 / 200

Run 3 time: 2.902115 minutes



In [9]:
CLAMPbase_time_minutes <- CLAMPbase_times
names(CLAMPbase_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(CLAMPbase_time_minutes, file.path(output_dir, "CLAMPbase_time_minutes.rds"))
cat("CLAMPbase times:", CLAMPbase_time_minutes, "minutes\n")

CLAMPbase times: 2.988649 2.829937 2.902115 minutes


## CLAMPfull with BP prior (3 runs)

In [10]:
CLAMPfull_times <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("CLAMPfull run", i, "of", N_RUNS, "\n")
  
  set.seed(seeds[i])
  
  start_time <- Sys.time()
  
  gtex_fullRes <- CLAMPfull(
    Y                 = gtex_fbm_filt,
    priorMat          = as.matrix(gtex_matched),
    svdres            = svd_results[[i]],
    clamp.base.result = base_results[[i]],
    clamp_k           = CLAMP_K_values[i],
    doCrossval        = TRUE,
    trace             = TRUE,
    use_cpp           = TRUE
  )
  
  end_time <- Sys.time()
  CLAMPfull_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", CLAMPfull_times[i], "minutes\n\n")
}

CLAMPfull run 1 of 3 


** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.277637028454; L2=135.832911085362

Progress 1 / 30 | Bdiff=0.000427

Progress 2 / 30 | Bdiff=0.038549



Progress 3 / 30 | Bdiff=0.015249

Estimated total runtime: ~7.1 min

Progress 4 / 30 | Bdiff=0.012160

Progress 5 / 30 | Bdiff=0.009770

Progress 6 / 30 | Bdiff=0.007010

Progress 7 / 30 | Bdiff=0.006031

Progress 8 / 30 | Bdiff=0.005255

Progress 9 / 30 | Bdiff=0.005009

Progress 10 / 30 | Bdiff=0.004579

Progress 11 / 30 | Bdiff=0.004519

Progress 12 / 30 | Bdiff=0.004554

Progress 13 / 30 | Bdiff=0.004239

Progress 14 / 30 | Bdiff=0.004223

Progress 15 / 30 | Bdiff=0.004037

Progress 16 / 30 | Bdiff=0.003654

Progress 17 / 30 | Bdiff=0.003570

Progress 18 / 30 | Bdiff=0.003246

Progress 19 / 30 | Bdiff=0.003327

Progress 20 / 30 | Bdiff=0.003406

Progress 21 / 30 | Bdiff=0.003378

Progress 22 / 30 | Bdiff=0.003376

Progress 23 / 30 | Bdiff=0.003460

Progress 24 / 30 | Bdiff=0.003425

Progress 25 / 30 | Bdiff=0.003423

Progress 26 / 30 | Bdiff=0.003409

Progress 27 / 30 | Bdiff=0.003180

Progress 28 / 30 | Bdiff=0.003352

Progress 29 / 30 | Bdiff=0.003527

Progress 30 / 30 | Bdiff=0.

Run 1 time: 12.04986 minutes

CLAMPfull run 2 of 3 


** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2776576560399; L2=135.83297296812

Progress 1 / 30 | Bdiff=0.000434

Progress 2 / 30 | Bdiff=0.037976

Progress 3 / 30 | Bdiff=0.015124

Estimated total runtime: ~6.9 min

Progress 4 / 30 | Bdiff=0.012519

Progress 5 / 30 | Bdiff=0.008877

Progress 6 / 30 | Bdiff=0.006788

Progress 7 / 30 | Bdiff=0.005554

Progress 8 / 30 | Bdiff=0.005161

Progress 9 / 30 | Bdiff=0.004855

Progress 10 / 30 | Bdiff=0.004414

Progress 11 / 30 | Bdiff=0.004733

Progress 12 / 30 | Bdiff=0.005173

Progress 13 / 30 | Bdiff=0.005117

Progress 14 / 30 | Bdiff=0.004966

Progress 15 / 30 | Bdiff=0.004597

Progress 16 / 30 | Bdiff=0.004085

Progress 17 / 30 | Bdiff=0.003875

Progress 18 / 30 | Bdiff=0.003481

Progress 19 / 30 | Bdiff=0.003349

Progress 20 / 30 | Bdiff=0.003229

Progress 21 / 30 | Bdiff=0.003023

Progress 22 / 30 | Bdiff=0.003157

Progress 23 / 30 | Bdiff=0.003189

Progress 24 / 30 | Bdiff=0.003252

Progress 25 / 30 | B

Run 2 time: 12.12031 minutes

CLAMPfull run 3 of 3 


** CLAMPfull **

using provided CLAMPbase result

CLAMP k is set to 412

L1=45.2776247586925; L2=135.832874276078

Progress 1 / 30 | Bdiff=0.000429

Progress 2 / 30 | Bdiff=0.037940

Progress 3 / 30 | Bdiff=0.015286

Estimated total runtime: ~6.8 min

Progress 4 / 30 | Bdiff=0.012514

Progress 5 / 30 | Bdiff=0.009134

Progress 6 / 30 | Bdiff=0.006728

Progress 7 / 30 | Bdiff=0.005717

Progress 8 / 30 | Bdiff=0.005207

Progress 9 / 30 | Bdiff=0.004925

Progress 10 / 30 | Bdiff=0.004653

Progress 11 / 30 | Bdiff=0.004656

Progress 12 / 30 | Bdiff=0.004619

Progress 13 / 30 | Bdiff=0.004358

Progress 14 / 30 | Bdiff=0.004136

Progress 15 / 30 | Bdiff=0.003931

Progress 16 / 30 | Bdiff=0.003682

Progress 17 / 30 | Bdiff=0.003586

Progress 18 / 30 | Bdiff=0.003290

Progress 19 / 30 | Bdiff=0.003404

Progress 20 / 30 | Bdiff=0.003309

Progress 21 / 30 | Bdiff=0.003215

Progress 22 / 30 | Bdiff=0.003124

Progress 23 / 30 | Bdiff=0.003197

Progress 24 / 30 | Bdiff=0.003160

Progress 25 / 30 | 

Run 3 time: 12.08806 minutes



In [11]:
CLAMPfull_BP_time_minutes <- CLAMPfull_times
names(CLAMPfull_BP_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(CLAMPfull_BP_time_minutes, file.path(output_dir, "CLAMPfull_BP_time_minutes.rds"))
cat("CLAMPfull (BP) times:", CLAMPfull_BP_time_minutes, "minutes\n")

CLAMPfull (BP) times: 12.04986 12.12031 12.08806 minutes


## Summary

In [12]:
cat("\n=== Timing Summary (minutes) ===\n")
cat("SVD:       ", paste(round(SVD_time_minutes, 2), collapse = ", "), "\n")
cat("CLAMPbase: ", paste(round(CLAMPbase_time_minutes, 2), collapse = ", "), "\n")
cat("CLAMPfull: ", paste(round(CLAMPfull_BP_time_minutes, 2), collapse = ", "), "\n")
cat("\nTotal per run:", paste(round(SVD_time_minutes + CLAMPbase_time_minutes + CLAMPfull_BP_time_minutes, 2), collapse = ", "), "\n")


=== Timing Summary (minutes) ===
SVD:        23.06, 22.97, 23.64 
CLAMPbase:  2.99, 2.83, 2.9 
CLAMPfull:  12.05, 12.12, 12.09 

Total per run: 38.1, 37.92, 38.64 
